# Retinal bottleneck in anatomically constrained CNNs

Deep Learning project work (Prof. A. Asperti). Reduced-scale reproduction of Lindsey, Ocko, Ganguli, Deny, *A unified theory of early visual representations from retina to cortex through anatomically constrained deep CNNs* (ICLR 2019).

**Datasets.** STL-10 and Imagenette, both at 96x96.

**Hypotheses.**
1. A narrow bottleneck with a deep cortex makes the retina learn center-surround filters.
2. A wide bottleneck or a shallow cortex makes the retina learn oriented filters.
3. Accuracy degrades little as the bottleneck narrows, until a minimum width is reached.

**Ablations.** Bottleneck channels, retina depth, cortex depth. 3 seeds per configuration, on both datasets.

In [ ]:
import itertools, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms as T
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

SEEDS = [0, 1, 2]
DATASETS = ['stl10', 'imagenette']
IMG_SIZE = 96
GRAYSCALE = True  # as in the paper (grayscale CIFAR-10)
BOTTLENECKS = [1, 2, 4, 8, 16, 32]
RETINA_LAYERS = [1, 2]
CORTEX_LAYERS = [2, 4]
KERNEL = 9
EPOCHS = 20
BATCH_SIZE = 32
LR = 1e-3
INIT = 'glorot'   # paper: Glorot-uniform
OPT = 'rmsprop'   # paper: RMSProp (Keras defaults)
WEIGHT_DECAY = 0.0
SHIFT = 0         # random translation (px) of training images
DATA_ROOT = Path('data')
RESULTS_CSV = Path('results/results.csv')
FILTERS_CSV = Path('results/filters.csv')
CKPT_DIR = Path('checkpoints')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


## Data

10% of the training set is held out for validation (split depends on the seed). Test: STL-10 `test`, Imagenette `val`. Images are decoded once (resize to 96, center crop, grayscale) and kept as uint8 on the GPU; the only augmentation is a random horizontal flip.


In [ ]:
class GPULoader:
    """Whole split as a uint8 tensor on the device; yields float batches in [0, 1]. Train: random flip and translation of up to `shift` px."""
    def __init__(self, x, y, shuffle, flip=False, shift=0):
        self.x, self.y, self.shuffle, self.flip, self.shift = x.to(DEVICE), y.to(DEVICE), shuffle, flip, shift

    def __len__(self):
        return -(-len(self.y) // BATCH_SIZE)

    def __iter__(self):
        order = torch.randperm(len(self.y), device=DEVICE) if self.shuffle else torch.arange(len(self.y), device=DEVICE)
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            x = self.x[idx].float() / 255
            if self.flip:
                x = torch.where(torch.rand(len(idx), 1, 1, 1, device=DEVICE) < 0.5, x.flip(3), x)
            if self.shift:
                theta = torch.zeros(len(idx), 2, 3, device=DEVICE)
                theta[:, 0, 0] = theta[:, 1, 1] = 1
                theta[:, :, 2] = (torch.rand(len(idx), 2, device=DEVICE) * 2 - 1) * self.shift * 2 / x.shape[-1]
                x = F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False), padding_mode='reflection', align_corners=False)
            yield x, self.y[idx]

_RAW = {}

def raw_splits(name):
    """Decode once: Resize -> CenterCrop -> (Grayscale), kept as uint8."""
    key = (name, IMG_SIZE)
    if key not in _RAW:
        tf = T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE)] + ([T.Grayscale()] if GRAYSCALE else []) + [T.PILToTensor()])
        root = DATA_ROOT / name
        if name == 'stl10':
            splits = [torchvision.datasets.STL10(root, split=s, transform=tf, download=True) for s in ('train', 'test')]
        elif name == 'imagenette':
            splits = [torchvision.datasets.Imagenette(root, split=s, size='160px', transform=tf, download=True) for s in ('train', 'val')]
        else:
            raise ValueError(name)
        _RAW[key] = [(torch.stack([x for x, _ in ds]), torch.tensor([y for _, y in ds])) for ds in splits]
    return _RAW[key]

def get_loaders(name, seed):
    (x_tr, y_tr), (x_te, y_te) = raw_splits(name)
    n = len(y_tr)
    n_val = n // 10
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(seed))
    tr, va = perm[n_val:], perm[:n_val]
    return (GPULoader(x_tr[tr], y_tr[tr], True, flip=True, shift=SHIFT), GPULoader(x_tr[va], y_tr[va], False),
            GPULoader(x_te, y_te, False), (1 if GRAYSCALE else 3))


## Model

Retina: `retina_layers` convolutions whose last layer has `bottleneck` channels. Cortex: `cortex_layers` convolutions with max pooling, then global average pooling and a linear classifier.

**Comparison with the paper (checked against Lindsey et al. 2019, Sec. 2 and App. A).**

| | paper | here | reason |
|---|---|---|---|
| input | grayscale CIFAR-10, 32x32 | grayscale STL-10 / Imagenette, 96x96 | professor's request |
| kernel, stride | 9x9, 1 | 9x9, 1 | same |
| width | 32 channels, retina 32 -> N_BN | same | same |
| nonlinearity, normalisation | ReLU, no batch-norm | same | same |
| retina depth | 2 | 1 or 2 | ablation |
| cortex depth (D_VVS) | 0-4 conv, then FC 1024 + FC 10 | 2 or 4 conv, 2x2 max-pool after each, GAP, linear | at 96x96 a flatten + FC 1024 would have hundreds of millions of weights |
| optimiser | RMSProp, 20 epochs, batch 32 | RMSProp (rho 0.9, eps 1e-7), lr 1e-3, 20 epochs, batch 32, best-val checkpoint | same, plus checkpoint selection |
| initialisation | Glorot-uniform | Glorot-uniform, zero biases | same |
| input scaling | not stated | pixels in [0, 1], then (x - 0.5) / 0.25 inside the model | centred input |
| input scaling | not stated | pixels in [0, 1], then (x - 0.5) / 0.25 inside the model | helps training; Adam lr 3e-4 (1e-3 gave dead retina channels in a pilot, picked on validation accuracy) |
| RF probe | gradient at a blank image | gradient averaged over natural images | silent units on a flat image |

The first retina layer is not analysed: the paper reports no clear structure there.

**Training recipe history (validation accuracy and RF health only, no test data).** A first sweep used PyTorch defaults with Adam (lr 1e-3, then 3e-4): many retina channels were dead and the RFs were high-frequency noise (share of RF power above 0.25 cycles/pixel about 0.5, validation accuracy 0.36 for STL-10 at bottleneck 4). The paper's recipe (Glorot init, RMSProp, lr 1e-3, batch 32) gave 0.48, no dead channels at small bottlenecks and smooth RFs (high-frequency share about 0.1 or less). Translation augmentation, weight decay and 32x32 input did not help in the pilot. The recipe was therefore fixed to the paper's and the whole sweep was rerun; the first sweep is not used.


In [ ]:
class RetinaCortexNet(nn.Module):
    def __init__(self, in_ch, num_classes, bottleneck, retina_layers=2, cortex_layers=2, kernel=KERNEL, width=32):
        super().__init__()
        pad = kernel // 2
        retina, c = [], in_ch
        for i in range(retina_layers):
            out = bottleneck if i == retina_layers - 1 else width
            retina += [nn.Conv2d(c, out, kernel, padding=pad), nn.ReLU()]
            c = out
        cortex = []
        for _ in range(cortex_layers):
            cortex += [nn.Conv2d(c, width, kernel, padding=pad), nn.ReLU(), nn.MaxPool2d(2)]
            c = width
        self.retina = nn.Sequential(*retina)
        self.cortex = nn.Sequential(*cortex)
        self.head = nn.Linear(width, num_classes)
        if INIT == 'glorot':
            for m in self.modules():
                if isinstance(m, (nn.Conv2d, nn.Linear)):
                    nn.init.xavier_uniform_(m.weight)
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.cortex(self.retina((x - 0.5) / 0.25))
        return self.head(x.mean(dim=(2, 3)))

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = n = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model(x).argmax(1) == y).sum().item()
        n += len(y)
    return correct / n

def make_optimizer(model):
    if OPT == 'rmsprop':  # Keras defaults: rho 0.9, eps 1e-7
        return torch.optim.RMSprop(model.parameters(), lr=LR, alpha=0.9, eps=1e-7, weight_decay=WEIGHT_DECAY)
    return torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def train(model, train_dl, val_dl):
    opt = make_optimizer(model)
    best, best_state = -1.0, None
    for _ in range(EPOCHS):
        model.train()
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        acc = evaluate(model, val_dl)
        if acc > best:
            best = acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return best


## Experiments

Main sweep: bottleneck width with retina depth 2 and cortex depth 2. Depth ablation: all other depth combinations at bottleneck 2 and 16. The loop is resumable through the checkpoint files.

In [ ]:
CONFIGS = [dict(bottleneck=b, retina_layers=2, cortex_layers=2) for b in BOTTLENECKS]
CONFIGS += [
    dict(bottleneck=b, retina_layers=rl, cortex_layers=cl)
    for b in (2, 16) for rl in RETINA_LAYERS for cl in CORTEX_LAYERS if (rl, cl) != (2, 2)
]

def run_one(ds, cfg, seed):
    set_seed(seed)
    train_dl, val_dl, test_dl, in_ch = get_loaders(ds, seed)
    model = RetinaCortexNet(in_ch, 10, **cfg).to(DEVICE)
    val_acc = train(model, train_dl, val_dl)
    return model, dict(val_acc=val_acc, test_acc=evaluate(model, test_dl))

RESULTS_CSV.parent.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)
for ds, cfg, s in itertools.product(DATASETS, CONFIGS, SEEDS):
    bn, rl, cl = cfg['bottleneck'], cfg['retina_layers'], cfg['cortex_layers']
    tag = f'{ds}_b{bn}_r{rl}_c{cl}_s{s}'
    if (CKPT_DIR / f'{tag}.pt').exists():
        continue
    t0 = time.time()
    model, metrics = run_one(ds, cfg, s)
    torch.save(model.state_dict(), CKPT_DIR / f'{tag}.pt')
    row = dict(dataset=ds, seed=s, time_s=time.time() - t0, **cfg, **metrics)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=not RESULTS_CSV.exists(), index=False)

In [ ]:
# Training occasionally collapses to chance level (dead network). Rule fixed after the sweep: a run whose best validation accuracy
# is below COLLAPSE_VAL is retrained once with lr / 3; both runs are kept (raw results.csv, reruns.csv, checkpoints/collapsed/).
COLLAPSE_VAL = 0.2
RERUN_LR = LR / 3
RERUNS_CSV = Path('results/reruns.csv')
KEY = ['dataset', 'bottleneck', 'retina_layers', 'cortex_layers', 'seed']
raw = pd.read_csv(RESULTS_CSV)
rerun_done = set(map(tuple, pd.read_csv(RERUNS_CSV)[KEY].values)) if RERUNS_CSV.exists() else set()
(CKPT_DIR / 'collapsed').mkdir(exist_ok=True)
for _, r in raw[raw.val_acc < COLLAPSE_VAL].iterrows():
    cfg = dict(bottleneck=int(r.bottleneck), retina_layers=int(r.retina_layers), cortex_layers=int(r.cortex_layers))
    if (r.dataset, *cfg.values(), int(r.seed)) in rerun_done:
        continue
    tag = f"{r.dataset}_b{cfg['bottleneck']}_r{cfg['retina_layers']}_c{cfg['cortex_layers']}_s{int(r.seed)}"
    t0 = time.time()
    lr_orig, LR = LR, RERUN_LR
    try:
        model, metrics = run_one(r.dataset, cfg, int(r.seed))
    finally:
        LR = lr_orig
    (CKPT_DIR / f'{tag}.pt').rename(CKPT_DIR / 'collapsed' / f'{tag}.pt')
    torch.save(model.state_dict(), CKPT_DIR / f'{tag}.pt')
    row = dict(dataset=r.dataset, seed=int(r.seed), time_s=time.time() - t0, **cfg, **metrics, lr=RERUN_LR)
    pd.DataFrame([row]).to_csv(RERUNS_CSV, mode='a', header=not RERUNS_CSV.exists(), index=False)


## Accuracy vs bottleneck width

In [ ]:
df = pd.read_csv(RESULTS_CSV)
if RERUNS_CSV.exists():  # collapsed runs are replaced by their rerun (lr / 3)
    rr = pd.read_csv(RERUNS_CSV)
    df = pd.concat([df.merge(rr[KEY], on=KEY, how='left', indicator=True).query("_merge == 'left_only'").drop(columns='_merge'),
                    rr.drop(columns='lr')], ignore_index=True)
keys = ['dataset', 'bottleneck', 'retina_layers', 'cortex_layers']
summary = df.groupby(keys).test_acc.agg(['mean', 'std']).round(4)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, ds in zip(axes, DATASETS):
    g = summary.xs((ds, 2, 2), level=['dataset', 'retina_layers', 'cortex_layers'])
    ax.errorbar(g.index, g['mean'], g['std'], marker='o', capsize=3)
    ax.set(xscale='log', xticks=BOTTLENECKS, xticklabels=BOTTLENECKS, title=ds, xlabel='bottleneck channels')
    ax.minorticks_off()
axes[0].set_ylabel('test accuracy')
plt.show()

# H3: accuracy drop w.r.t. the widest bottleneck, and the narrowest width within TOL of it.
TOL = 0.01
main = summary.xs((2, 2), level=['retina_layers', 'cortex_layers'])['mean'].unstack('dataset')
drop = (main.loc[BOTTLENECKS[-1]] - main).round(4)
print('accuracy drop vs bottleneck', BOTTLENECKS[-1])
display(drop)
print(f'smallest bottleneck within {TOL:.0%} of the widest:', (drop <= TOL).idxmax().to_dict())
summary


## Learned filters

Receptive field (RF) of each retina output channel (`retina_layers * 8 + 1` pixels wide): gradient of the channel's activation at the image centre with respect to the input, averaged over 256 natural images. This is the paper's linear RF estimate except for the probe: the paper uses a blank image, but in our networks most retina units are silent on a flat image (a check on a trained net gave 7 silent channels out of 8), so the blank-image gradient is zero. Each RF is classified with two tools:

- **Orientation selectivity index (OSI)**, as in App. A of the paper: strongest response to a rectangular bar at the preferred orientation divided by the strongest response at the orthogonal one (about 1 for isotropic RFs, larger for oriented ones; the paper reports 1.56 for the retina output, 3.05 for the first cortical layer, 1.97 for random RFs).
- **Fit quality ($R^2$) of an isotropic difference of Gaussians (DoG) and of a Gabor**.

Label: `center-surround` if OSI < `OSI_THR` and the DoG fits well (not worse than the Gabor by more than `R2_TOL`); `oriented` if OSI >= `OSI_THR` and the Gabor fits better than the DoG; `dead` if the gradient is zero; `other` otherwise. Thresholds are set on synthetic filters only, before looking at learned ones.


In [ ]:
def load_model(ds, cfg, seed):
    tag = f"{ds}_b{cfg['bottleneck']}_r{cfg['retina_layers']}_c{cfg['cortex_layers']}_s{seed}"
    model = RetinaCortexNet(1 if GRAYSCALE else 3, 10, **cfg)
    model.load_state_dict(torch.load(CKPT_DIR / f'{tag}.pt', map_location='cpu'))
    return model.to(DEVICE).eval()

def probe_images(ds, n=256):
    x = raw_splits(ds)[1][0]
    idx = torch.randperm(len(x), generator=torch.Generator().manual_seed(0))[:n]
    return x[idx].float().div(255).to(DEVICE)

def retina_rfs(model, images):
    """Linear RF of each retina output channel at the image centre: gradient w.r.t. the input, averaged over natural images.
    The paper uses a blank image; in our nets most units are silent on a flat image (ReLU, negative biases), so the blank-image gradient is 0."""
    n_conv = sum(isinstance(m, nn.Conv2d) for m in model.retina)
    c, h = IMG_SIZE // 2, n_conv * (KERNEL - 1) // 2
    x = images.clone().requires_grad_(True)
    out = model.retina(x)
    rfs = []
    for ch in range(out.shape[1]):
        g, = torch.autograd.grad(out[:, ch, c, c].sum(), x, retain_graph=True)
        rfs.append((g.mean(0).mean(0) )[c - h:c + h + 1, c - h:c + h + 1].cpu().numpy())
    return np.stack(rfs)

def plot_filters(rfs, labels=None, ncols=8, title=''):
    nrows = int(np.ceil(len(rfs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.5, nrows * 1.6), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for i, (ax, f) in enumerate(zip(axes.ravel(), rfs)):
        m = max(np.abs(f).max(), 1e-12)
        ax.imshow(f, cmap='RdBu', vmin=-m, vmax=m)
        if labels is not None:
            ax.set_title(labels[i], fontsize=7)
    fig.suptitle(title)
    plt.show()


In [ ]:
OSI_THR, R2_MIN, R2_TOL = 2.0, 0.5, 0.05
LABELS = ['center-surround', 'oriented', 'other', 'dead']

def _grid(n):
    yy, xx = np.mgrid[:n, :n].astype(float) - (n - 1) / 2
    return xx, yy

def orientation_index(rf, n_orient=12, widths=(1, 2, 3, 4, 5, 6)):
    """Paper App. A: best bar response at the preferred orientation / best response at the orthogonal one."""
    n = rf.shape[0]
    xx, yy = _grid(n)
    offsets = np.arange(-(n // 2), n // 2 + 1)
    resp = np.zeros((len(widths), n_orient, len(offsets)))
    for j in range(n_orient):
        th = j * np.pi / n_orient
        u = xx * np.cos(th) + yy * np.sin(th)
        for i, w in enumerate(widths):
            bars = np.abs(u[None] - offsets[:, None, None]) <= w / 2
            resp[i, j] = np.abs(bars.reshape(len(offsets), -1) @ rf.ravel())
    i, j, _ = np.unravel_index(resp.argmax(), resp.shape)
    s = resp[i].max(axis=1)
    perp = s[(j + n_orient // 2) % n_orient]
    return s[j] / max(perp, 1e-8 * s[j]), j * np.pi / n_orient

def _dog(p, xx, yy):
    x0, y0, sc, ss, a, k = p
    r2 = (xx - x0) ** 2 + (yy - y0) ** 2
    return a * (np.exp(-r2 / (2 * sc ** 2)) - k * np.exp(-r2 / (2 * ss ** 2)))

def _gabor(p, xx, yy):
    x0, y0, sx, sy, th, f, ph, a = p
    u = (xx - x0) * np.cos(th) + (yy - y0) * np.sin(th)
    v = -(xx - x0) * np.sin(th) + (yy - y0) * np.cos(th)
    return a * np.exp(-u ** 2 / (2 * sx ** 2) - v ** 2 / (2 * sy ** 2)) * np.cos(2 * np.pi * f * u + ph)

def _best_r2(fn, starts, lo, hi, rf, xx, yy):
    sst = ((rf - rf.mean()) ** 2).sum()
    best = -np.inf
    for p0 in starts:
        res = least_squares(lambda p: (fn(p, xx, yy) - rf).ravel(), p0, bounds=(lo, hi), max_nfev=200)
        best = max(best, 1 - (2 * res.cost) / sst)
    return best

def filter_stats(rf):
    n = rf.shape[0]
    if np.abs(rf).max() < 1e-10:
        return dict(osi=np.nan, r2_dog=np.nan, r2_gabor=np.nan, label='dead')
    rf = rf / np.abs(rf).max()
    xx, yy = _grid(n)
    osi, th = orientation_index(rf)
    sign = 1.0 if rf.flat[np.abs(rf).argmax()] > 0 else -1.0
    s = n / 10
    r2_dog = _best_r2(
        _dog, [[0, 0, sc, 2.5 * sc, sign, 0.8] for sc in (s, 1.6 * s)],
        [-n / 4, -n / 4, 0.7, 0.7, -3, 0.2], [n / 4, n / 4, n / 2, n, 3, 2], rf, xx, yy)
    r2_gabor = _best_r2(
        _gabor, [[0, 0, n / 5, n / 5, t, f, ph, sign] for t in (th, th + np.pi / 2) for f in (0.1, 0.2) for ph in (0, np.pi / 2)],
        [-n / 3, -n / 3, 0.7, 0.7, -2 * np.pi, 0.05, -2 * np.pi, -3], [n / 3, n / 3, n, n, 2 * np.pi, 0.5, 2 * np.pi, 3], rf, xx, yy)
    if osi < OSI_THR and r2_dog >= R2_MIN and r2_dog >= r2_gabor - R2_TOL:
        label = 'center-surround'
    elif osi >= OSI_THR and r2_gabor >= R2_MIN and r2_gabor > r2_dog:
        label = 'oriented'
    else:
        label = 'other'
    return dict(osi=osi, r2_dog=r2_dog, r2_gabor=r2_gabor, label=label)

def filter_type(kernel_2d):
    return filter_stats(np.asarray(kernel_2d, dtype=float))['label']


In [ ]:
# Sanity check of the classifier on synthetic 17x17 filters (ground truth known), with 10% additive noise.
rng = np.random.default_rng(0)
xx, yy = _grid(17)
def synth(kind):
    if kind == 'center-surround':
        sc = rng.uniform(1.5, 3)
        rf = _dog([rng.uniform(-1, 1), rng.uniform(-1, 1), sc, sc * rng.uniform(1.8, 3), rng.choice([-1, 1]), rng.uniform(0.6, 1.0)], xx, yy)
    elif kind == 'oriented':
        rf = _gabor([rng.uniform(-1, 1), rng.uniform(-1, 1), rng.uniform(2.5, 4.5), rng.uniform(2.5, 4.5), rng.uniform(0, np.pi),
                     rng.uniform(0.08, 0.2), rng.uniform(0, 2 * np.pi), 1], xx, yy)
    else:
        return rng.standard_normal((17, 17))
    return rf / np.abs(rf).max() + 0.1 * rng.standard_normal((17, 17))

check = pd.DataFrame([dict(truth=k, pred=filter_type(synth(k))) for k in ['center-surround', 'oriented', 'noise'] for _ in range(40)])
pd.crosstab(check.truth, check.pred)


In [ ]:
def analyse_filters(only=None):
    rows = []
    for ds in DATASETS:
        images = probe_images(ds)
        for cfg, s in itertools.product(CONFIGS, SEEDS):
            if only is not None and (ds, cfg['bottleneck'], cfg['retina_layers'], cfg['cortex_layers'], s) not in only:
                continue
            for ch, rf in enumerate(retina_rfs(load_model(ds, cfg, s), images)):
                rows.append(dict(dataset=ds, seed=s, channel=ch, **cfg, **filter_stats(rf)))
    return pd.DataFrame(rows)

if FILTERS_CSV.exists():
    filters_df = pd.read_csv(FILTERS_CSV)
    if RERUNS_CSV.exists():  # refresh only the networks that were retrained after a collapse
        stale = set(map(tuple, pd.read_csv(RERUNS_CSV)[KEY].values))
        fresh = analyse_filters(stale)
        old = filters_df.merge(fresh[KEY].drop_duplicates(), on=KEY, how='left', indicator=True)
        filters_df = pd.concat([old.query("_merge == 'left_only'").drop(columns='_merge'), fresh], ignore_index=True)
        filters_df.to_csv(FILTERS_CSV, index=False)
else:
    filters_df = analyse_filters()
    filters_df.to_csv(FILTERS_CSV, index=False)
filters_df.groupby('label').size()


In [ ]:
def show_rfs(ds, cfg, seed=0):
    rfs = retina_rfs(load_model(ds, cfg, seed), probe_images(ds))
    labels = [filter_type(rf) for rf in rfs]
    plot_filters(rfs, labels, title=f"{ds}, seed {seed}: bottleneck {cfg['bottleneck']}, retina {cfg['retina_layers']}, cortex {cfg['cortex_layers']}")

for b in (1, 4, 32):
    show_rfs(DATASETS[0], dict(bottleneck=b, retina_layers=2, cortex_layers=2))


In [ ]:
per_seed = (filters_df.groupby(keys + ['seed']).label.value_counts(normalize=True).unstack(fill_value=0)
            .reindex(columns=LABELS, fill_value=0))
per_seed['osi'] = filters_df.groupby(keys + ['seed']).osi.mean()
per_cfg = per_seed.groupby(keys).agg(['mean', 'std'])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, ds in zip(axes, DATASETS):
    g = per_cfg.xs((ds, 2, 2), level=['dataset', 'retina_layers', 'cortex_layers'])
    for lab in LABELS[:3]:
        ax.errorbar(g.index, g[(lab, 'mean')], g[(lab, 'std')], marker='o', capsize=3, label=lab)
    ax.set(xscale='log', xticks=BOTTLENECKS, xticklabels=BOTTLENECKS, title=ds, xlabel='bottleneck channels')
    ax.minorticks_off()
axes[0].set_ylabel('fraction of retina channels')
axes[0].legend()
plt.show()

per_cfg.round(2)


In [ ]:
# Depth ablation at bottleneck 2 and 16 (mean over seeds): accuracy, share of center-surround / oriented channels, mean OSI.
depth = (summary['mean'].rename('acc').to_frame()
         .join(per_seed.groupby(keys)[['center-surround', 'oriented', 'osi']].mean())
         .loc[lambda d: d.index.get_level_values('bottleneck').isin([2, 16])]
         .sort_index(level=['dataset', 'bottleneck', 'retina_layers', 'cortex_layers']).round(3))
depth


## Discussion

For each hypothesis: expected outcome, observed outcome, explanation of any mismatch with the paper. Numbers: mean over 3 seeds, 96x96 grayscale, retina depth 2 and cortex depth 2 unless stated; shares are fractions of retina output channels by label (702 channels in total: 290 `other`, 264 `oriented`, 137 `dead`, 11 `center-surround`). The OSI averaged over non-dead channels is 3.5 (median 2.6); the paper reports 1.56 for the retina output, 3.05 for the first cortical layer and 1.97 for random filters.

- **H1 (narrow bottleneck + deep cortex -> center-surround retina filters): not reproduced.**
  - Only 11 of 702 channels (1.6%) are classified as center-surround, and none in the narrow-bottleneck, deep-cortex configuration that the hypothesis is about (bottleneck 2, cortex 4: 0% on both datasets). With a bottleneck of 1-2 channels and retina depth 2 the retina filters are mostly `oriented` (bottleneck 2, cortex 4: 50% on STL-10, 67% on Imagenette) and the retina OSI is 2.9-4.9, i.e. in the range the paper reports for cortex, not for the retina.
  - The paper's claim therefore does not hold in this reduced setting. We did not test why. Differences that could matter, none of which was isolated: far less training data (4.5k and 8.5k images instead of 50k), 96x96 instead of 32x32 inputs (the same 9x9 kernels cover a much smaller part of the scene; a 32x32 pilot also gave no center-surround filters), global average pooling plus a linear classifier instead of a flattened dense layer, 3 seeds instead of 10, and our RF probe (gradient averaged over natural images instead of a blank image).
- **H2 (wide bottleneck or shallow cortex -> oriented retina filters): partly supported.**
  - Shallow cortex: with the bottleneck fixed, cortex depth 2 gives more oriented filters than cortex depth 4 in all four comparisons (bottleneck 2: STL-10 83% vs 50%, Imagenette 100% vs 67%; bottleneck 16: STL-10 50% vs 33%, Imagenette 56% vs 17%). Cortex 4 mostly turns the difference into `other` and `dead`, not center-surround, so this agrees with the paper only in direction (a deeper cortex makes the retina less oriented), not with "shallow cortex -> oriented, deep cortex -> center-surround".
  - Wide bottleneck: bottleneck 32 has 57% (STL-10) and 48% (Imagenette) oriented channels, but narrow bottlenecks are not different or are *more* oriented (bottleneck 1: 100% and 67%), so the expected bottleneck effect on filter geometry is absent or opposite.
  - Retina depth 1: most filters are `other` (60-100%); the 9x9 receptive field is small for the DoG/Gabor fits, so this label is not reliable.
- **H3 (accuracy drops little while the bottleneck stays above a threshold): not confirmed as a threshold; accuracy decreases gradually.**
  - Test accuracy for bottleneck 1, 2, 4, 8, 16, 32: STL-10 0.417, 0.462, 0.455, 0.496, 0.493, 0.519; Imagenette 0.553, 0.588, 0.616, 0.624, 0.643, 0.663. Doubling the bottleneck changes accuracy by -1 to +4.5 points step by step (the steps 2 to 4 and 8 to 16 are within noise), with no knee; overall, going from 32 to 1 channel costs about 10 points on both datasets. A "within 1 point of the widest" rule selects only the widest bottleneck on both datasets.
  - The paper also reports accuracy increasing with both bottleneck width and cortex depth; the cortex part is reproduced (bottleneck 2: cortex 4 vs 2 gives +12 points on STL-10 and +9 on Imagenette; bottleneck 16: +6 and +4).

### Limitations

- Absolute accuracy is low (0.42-0.71) because of the small training sets and the small network, and 20% of the retina channels are dead (never active), so the analysed filters are a subset of the trained ones.
- Training recipe: the first sweep used PyTorch defaults with Adam and produced dead and noisy filters; the recipe was then fixed to the paper's (Glorot, RMSProp, lr 1e-3, batch 32) after a validation-only pilot that also looked at filter health. 3 of 72 runs collapsed to chance level and were retrained once with lr / 3 (rule fixed after the sweep, raw results kept in `results.csv`). All reported numbers use the rerun values for those three runs.
- Filter labels depend on thresholds set on synthetic filters (100% accuracy there) and not validated against human labels on learned filters; the RF probe differs from the paper's blank-image gradient (silent units).
- Only 3 seeds (paper: 10), one architecture family, no ablation on kernel size or on the number of cortex channels.
